# 74. 交互热力图（px.imshow）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 11 / 18 步：表达层级、流程、贡献与地域**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 交互小提琴图（px.violin）  →  **本章任务：** 交互热力图（px.imshow）  →  **下一步：** 矩形树图（px.treemap）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

当我们想一眼看出「哪个区域、哪个品类卖得最好」，把数据排成一格一格的矩阵，用颜色深浅代表数值大小，比逐行翻表格高效得多。



## 本章目标

学完本章，你将能够：

- **理解**：理解「交互热力图（px.imshow）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互热力图（px.imshow）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互热力图（px.imshow）」并读出其中的结论。


## 74.1 适用场景

比较行×列数值矩阵或相关关系。

**背景引入**：

当我们想一眼看出「哪个区域、哪个品类卖得最好」，把数据排成一格一格的矩阵，用颜色深浅代表数值大小，比逐行翻表格高效得多。这种行×列填色图就是热力图，尤其适合做地区×品类、机场×月份这类交叉对比。Plotly 的 px.imshow 把静态热力图升级成交互式：鼠标悬停能精确读到每个格子里的数值，还可以缩放、平移，在给别人汇报发现时也更容易看懂。（可以把它想成一张“温度地图”：行是一个维度、列是另一个维度，每个格子的颜色深浅就是格子里数值的大小；颜色只是给格子“涂温度”，真正的数值要靠 hover 点上去读，所以先看颜色标尺再点格子。）


## 74.2 数据结构

二维数组或透视后的DataFrame。


## 74.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 color_continuous_scale="Blues" 改为 "Viridis" 或 "YlOrRd"，对比不同色盘的视觉效果
2. 修改 text_auto=".0f" 为 text_auto=False，观察单元格标注对精确读值的作用
3. 添加 zmin 和 zmax 固定色阶范围，说明固定色阶对跨图比较的意义


## 74.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `orders.pivot_table()`、`px.imshow()`、`fig.update_layout()`、`fig.show()` | 比较行×列数值矩阵或相关关系。 | 色阶被异常值拉伸 |
| 进阶变体 | `px.imshow()`、`fig.update_layout()`、`fig.show()`、`.corr()` | 在基础图表上增加分组、注释、布局或交互 | 发散数据使用单向色盘 |
| 关键参数 | `color_continuous_scale` | 色盘 | 色阶被异常值拉伸 |
| 关键参数 | `zmin/zmax` | 色阶 | 发散数据使用单向色盘 |
| 关键参数 | `text_auto` | 标注 | 行列顺序无业务逻辑 |
| 关键参数 | `aspect` | 宽高 | 色阶被异常值拉伸 |


## 74.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-74 -->
### 数学推导｜矩阵颜色必须对应明确的数值变换

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜按列估计中心与尺度。** $\mu_j=\frac1n\sum_ix_{ij}$，$\sigma_j^2=\frac1{n-1}\sum_i(x_{ij}-\mu_j)^2$。

**第 2 步｜把原值换成离均值多少个标准差。** $z_{ij}=(x_{ij}-\mu_j)/\sigma_j$。

**第 3 步｜检查变换结果。** 对非零方差列，标准化后近似满足

$$
\frac1n\sum_i z_{ij}\approx0,
\qquad
\frac1{n-1}\sum_i z_{ij}^2=1
$$

因此不同原始单位可以共享色阶，但颜色不再表示原单位。

**把上面的关系收束为本章计算式：**

$$
z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}
$$

**符号解释：** $z_{ij}$ 是按列标准化后的值，使不同单位的列可在同一色阶比较。

**代码对应：** 根据问题选择原值、比例、相关系数或 z-score，再设置统一色阶。

**使用边界：** 标准化会丢失原单位；色阶中心、范围和缺失值颜色都必须说明。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(f"Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行")


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 74.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
matrix = orders.pivot_table(
    index="region",
    columns="category",
    values="sales",
    aggfunc="sum",
    fill_value=0,
)
fig = px.imshow(
    matrix,
    text_auto=".0f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="区域品类销售额",
)
fig.update_layout(
    xaxis_title="品类", yaxis_title="区域", coloraxis_colorbar_title="销售额"
)
fig.show()


**练一练**：下面的基础图表用 `color_continuous_scale="Blues"` 画了「区域 × 品类」销售额热力图。请你把色盘换一种（例如 `"Viridis"`、`"YlOrRd"`），再看看标注参数 `text_auto` 若设为 `False` 会有什么不同，并观察色盘改变后哪几个格子颜色变得最醒目。\n\n提示：只改一个参数、运行、观察，再改下一个；把观察到的变化写到注释里。\n


In [ ]:
# 请在下方填写代码
# 目标：用 px.imshow 画出「区域 × 品类」销售额热力图，并更换色盘观察变化。
# 1. 构造行×列矩阵（把 ___ 换成正确字段）
matrix = orders.pivot_table(
    index="___",  # 行取区域字段：region
    columns="___",  # 列取品类字段：category
    values="___",  # 数值字段：sales
    aggfunc="sum",
    fill_value=0,
)
# 2. 绘制热力图，并把色盘换成你选的一种
fig = px.imshow(
    matrix,
    text_auto=".0f",
    color_continuous_scale="___",  # 试试 "Viridis" / "YlOrRd" 等
    aspect="auto",
)


In [ ]:
# 答案：画「区域 × 品类」销售额热力图，并把色盘改为 Viridis
matrix = orders.pivot_table(
    index="region",
    columns="category",
    values="sales",
    aggfunc="sum",
    fill_value=0,
)
fig = px.imshow(
    matrix,
    text_auto=".0f",
    color_continuous_scale="Viridis",
    aspect="auto",
    title="区域品类销售额（Viridis）",
)
fig.update_layout(
    xaxis_title="品类", yaxis_title="区域", coloraxis_colorbar_title="销售额"
)


## 74.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
corr = orders[["order_value", "items", "sales"]].corr()
fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="订单指标相关矩阵",
)
fig.update_layout(coloraxis_colorbar_title="相关系数")
fig.show()


## 74.8 参数说明

- color_continuous_scale：色盘
- zmin/zmax：色阶
- text_auto：标注
- aspect：宽高


## 74.9 结果解读

先理解颜色范围，再通过Hover确认极值单元格。


## 74.10 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig = px.bar(report, x="region", y="sales", title="地区销售额")
fig.show()


### 74.10.1 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
fig.show()


### 74.10.2 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 74.11 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 74.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 74.12 易错点提醒

- 色阶被异常值拉伸
- 发散数据使用单向色盘
- 行列顺序无业务逻辑


## 74.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 74.14 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：换一种色盘，观察色阶对读图的影响
# 【目标】色盘决定高值低值的颜色，练习体会冷暖色盘对强调差异的影响。
import plotly.express as px

# 起点示例(已可运行)：色盘换成 YlOrRd，比较对高值的强调差异。
matrix = orders.pivot_table(
    index="region", columns="category", values="sales", aggfunc="sum", fill_value=0
)
fig = px.imshow(
    matrix, text_auto=".0f", color_continuous_scale="YlOrRd", aspect="auto", title="区域品类销售额（YlOrRd）"
)
fig.update_layout(xaxis_title="品类", yaxis_title="区域", coloraxis_colorbar_title="销售额")
fig.show()

# ---- 反思记录：冷暖色盘对高值区的强调有何不同 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
channel_region = orders.pivot_table(
    index="region", columns="channel", values="order_value", aggfunc="mean"
)
fig = px.imshow(
    channel_region,
    text_auto=".1f",
    color_continuous_scale="YlGnBu",
    aspect="auto",
    title="区域渠道平均客单价",
)
fig.update_layout(xaxis_title="渠道", yaxis_title="区域")
fig.show()


## 74.15 小结

用交互热力图展示二维矩阵，并通过Hover读取精确行列组合。


### 74.15.1 你已经掌握

- 判断交互热力图（px.imshow）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 74.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `color_continuous_scale` | 色盘 |
| `zmin/zmax` | 色阶 |
| `text_auto` | 标注 |
| `aspect` | 宽高 |


### 74.15.3 需要注意

- 色阶被异常值拉伸
- 发散数据使用单向色盘
- 行列顺序无业务逻辑


### 74.15.4 完成检查

- [ ] 能判断什么问题适合使用交互热力图（px.imshow）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 74.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
